In [159]:
!pip install pyserial

In [160]:
import serial, time
!pip install pyserial

In [161]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [162]:
print(serial)

<module 'serial' from 'C:\\Users\\caleb\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [163]:
print(serial.__file__)

C:\Users\caleb\anaconda3\Lib\site-packages\serial\__init__.py


In [164]:
print(serial.__version__)

3.5


In [165]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [166]:
baudrate = 115200

In [167]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [168]:
#ser.close()

In [169]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [170]:
ser.in_waiting

31

In [171]:
#ser.close()

In [172]:
#read_all(ser)

In [173]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [174]:
read_all(ser)

'dual servo control over serial\n'

In [175]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [176]:
read_one_line(ser)

''

In [177]:
read_all(ser)

''

In [178]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [179]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

## Example

In [180]:
#byte1 = 7
#WriteByte(ser,byte1)#<--
#time.sleep(0.1)
#byte2 = 156
#WriteByte(ser,byte2)#<--
#time.sleep(0.1)
#next_line = read_one_line(ser)
#extra = read_all(ser)
#print('next_line: %s' % next_line)
#print('extra: %s' % extra)

# Break an integer into two bytes

In [181]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [182]:
# inputs from user
xll = 25 # x origin
yll = 25 # y origin
w = 10   # width
h = 10   # height
N = 10  # number of steps per side

In [183]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N, xll)  
y_left = np.linspace(yll+h, yll, N)

#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[25.        , 25.        ],
       [26.        , 25.        ],
       [27.        , 25.        ],
       [28.        , 25.        ],
       [29.        , 25.        ],
       [30.        , 25.        ],
       [31.        , 25.        ],
       [32.        , 25.        ],
       [33.        , 25.        ],
       [34.        , 25.        ],
       [35.        , 25.        ],
       [35.        , 26.        ],
       [35.        , 27.        ],
       [35.        , 28.        ],
       [35.        , 29.        ],
       [35.        , 30.        ],
       [35.        , 31.        ],
       [35.        , 32.        ],
       [35.        , 33.        ],
       [35.        , 34.        ],
       [35.        , 35.        ],
       [34.        , 35.        ],
       [33.        , 35.        ],
       [32.        , 35.        ],
       [31.        , 35.        ],
       [30.        , 35.        ],
       [29.        , 35.        ],
       [28.        , 35.        ],
       [27.        ,

In [184]:
#define link lengths
l1 = 25 # base link
l2 = 25 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
print(alpha_temp)
alpha = np.arccos(alpha_temp)

print(alpha*rtd)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

##theta2 = 180 - theta2

print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)

[-0.         -0.0408     -0.0832     -0.1272     -0.1728     -0.22
 -0.2688     -0.3192     -0.3712     -0.4248     -0.48       -0.5208
 -0.5632     -0.6072     -0.6528     -0.7        -0.7488     -0.7992
 -0.8512     -0.9048     -0.96       -0.9048     -0.8512     -0.7992
 -0.7488     -0.7        -0.6528     -0.6072     -0.5632     -0.5208
 -0.48       -0.41876543 -0.35950617 -0.30222222 -0.24691358 -0.19358025
 -0.14222222 -0.09283951 -0.0454321  -0.        ]
[ 90.          92.33831685  94.77252579  97.30782082  99.95065705
 102.70903299 105.59287234 108.61455118 111.78964344 115.13800472
 118.68540201 121.38592923 124.27738924 127.38731894 130.75304405
 134.427004   138.48653684 143.05377574 148.34242885 154.79635988
 163.73979529 154.79635988 148.34242885 143.05377574 138.48653684
 134.427004   130.75304405 127.38731894 124.27738924 121.38592923
 118.68540201 114.75666857 111.06987151 107.59112394 104.29494862
 101.16179749  98.17645714  95.32698301  92.60396383  90.        ]

thet

In [185]:
theta_min = 0
theta_max = 180
min_new = 1000
max_new = 2000

myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)


myint2 = min_new + (((180-(90+theta2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)

#myint2_dif = 1500 - myint2
#myint2 = 1500 - myint2_dif -500
print('\n',myint2)
#print('\n',myint2_dif)

#theta2_new = theta_min + ((myint - min_new) * (theta_max - theta_min)) / (max_new - min_new)
#theta2_new

[1000.         1000.25475396 1001.02035962 1002.30116729 1004.10518737
 1006.44493105 1009.33851446 1012.81115574 1016.89726898 1021.64349063
 1027.11321555 1040.55577459 1054.3684181  1068.63037624 1083.44831657
 1098.97109247 1115.41688423 1133.12843145 1152.70294503 1175.37702474
 1204.8327647  1184.60275237 1171.42165968 1161.61476713 1153.95276488
 1147.84559642 1142.95748371 1139.07695119 1136.0615221  1133.81049889
 1132.25012897 1116.4553098  1101.12077787 1086.14110433 1071.43370805
 1056.93198399 1042.58090525 1028.33410974 1014.15192324 1000.        ]

 [1000.         1012.99064918 1026.51403216 1040.59900453 1055.28142807
 1070.60573886 1086.62706855 1103.41417323 1121.05357465 1139.65558179
 1159.36334452 1174.36627348 1190.42994019 1207.70732743 1226.40580029
 1246.81668889 1269.36964911 1294.74319858 1324.12460471 1359.97977711
 1409.6655294  1359.97977711 1324.12460471 1294.74319858 1269.36964911
 1246.81668889 1226.40580029 1207.70732743 1190.42994019 1174.36627348
 11

In [186]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [187]:
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 3 3] 

 [232 232 233 234 236 238 241 244 248 253   3  16  30  44  59  74  91 109
 128 151 180 160 147 137 129 123 118 115 112 109 108  92  77  62  47  32
  18   4 246 232] 


 [3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 3 3] 

 [232 244   2  16  31  46  62  79  97 115 135 150 166 183 202 222 245  14
  44  79 129  79  44  14 245 222 202 183 166 150 135 113  93  73  55  38
  21   5 246 232]


In [188]:
# Send all path points to both servos
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    line1 = read_one_line(ser)  # servo 1 bytes echo
    line2 = read_one_line(ser)  # servo 1 int echo
    line3 = read_one_line(ser)  # servo 2 bytes echo
    line4 = read_one_line(ser)  # servo 2 int echo
    print(f"Step {i}: servo1={line2}  servo2={line4}")

    print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')

    time.sleep(1)  # pause between steps so servo has time to move

Step 0: servo1=1000  servo2=1000

theta1: 0.0 
theta2: 90.0 
 


Step 1: servo1=1000  servo2=1012

theta1: 0.04585571242542841 
theta2: 87.6616831469983 
 


Step 2: servo1=1001  servo2=1026

theta1: 0.18366473223974822 
theta2: 85.22747421198889 
 


Step 3: servo1=1002  servo2=1040

theta1: 0.41421011171893696 
theta2: 82.69217918435787 
 


Step 4: servo1=1004  servo2=1055

theta1: 0.7389337272711245 
theta2: 80.04934294734008 
 


Step 5: servo1=1006  servo2=1070

theta1: 1.1600875894629112 
theta2: 77.29096700560456 
 


Step 6: servo1=1009  servo2=1086

theta1: 1.6809326031715557 
theta2: 74.40712766108608 
 


Step 7: servo1=1012  servo2=1103

theta1: 2.30600803364554 
theta2: 71.38544881771826 
 


Step 8: servo1=1016  servo2=1121

theta1: 3.041508416968817 
theta2: 68.21035656210593 
 


Step 9: servo1=1021  servo2=1139

theta1: 3.895828313567044 
theta2: 64.86199527710637 
 


Step 10: servo1=1027  servo2=1159

theta1: 4.880378799033842 
theta2: 61.31459798588108 
 


Step 11

In [124]:
#byte3, byte4 = break_into_two(1200)

In [125]:
#WriteByte(ser, MSB)
#WriteByte(ser, LSB)

In [126]:

WriteByte(ser,byte1)#<--
time.sleep(0.1)

WriteByte(ser,byte2)#<--
time.sleep(0.1)

#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
byte3, byte4 = break_into_two(1500)

In [ ]:
WriteByte(ser,byte3)#<--
time.sleep(0.1)

WriteByte(ser,byte4)#<--
time.sleep(0.1)
#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

In [ ]:
ser.close()

In [ ]:
for i in range(1100, 1900, 50):
    byte3, byte4 = break_into_two(i)
    WriteByte(ser,byte3)#<--
    time.sleep(0.1)

    WriteByte(ser,byte4)#<--
    time.sleep(0.1)
    #WriteByte(ser,byte3)#<--
    #time.sleep(0.1)

    #WriteByte(ser,byte4)#<--
    #time.sleep(0.1)
    next_line = read_one_line(ser)
    extra = read_all(ser)
    print('next_line: %s' % next_line)
    print('extra: %s' % extra)
    time.sleep(0.5)
    print(i) 

- How do we break this into two bytes?
- How do we find the most significant byte?
- How do we find the least significant byte?

In [ ]:
ser.close()